In [1]:
"""
weather_utils.py
 
Utility functions for loading, cleaning, and querying weather CSV data
of the form: weatherdata-XXX-XXXX.csv
 
Each file contains columns:
    Date, Longitude, Latitude, Elevation, Max Temperature,
    Min Temperature, Precipitation, Wind, Relative Humidity, Solar
"""

'\nweather_utils.py\n\nUtility functions for loading, cleaning, and querying weather CSV data\nof the form: weatherdata-XXX-XXXX.csv\n\nEach file contains columns:\n    Date, Longitude, Latitude, Elevation, Max Temperature,\n    Min Temperature, Precipitation, Wind, Relative Humidity, Solar\n'

In [2]:
import glob
import os
from typing import Iterable, List, Sequence, Tuple
 
import pandas as pd

In [3]:
def load_weather_data(folder_path: str, pattern: str = "*.csv") -> pd.DataFrame:
    """
    Read all weather CSV files in a folder into a single pandas DataFrame.
 
    Parameters
    ----------
    folder_path : str
        Path to the folder containing the CSV files.
    pattern : str, optional
        Glob pattern used to select files (default matches
        'weatherdata-*.csv'). Adjust if your files are named differently.
 
    Returns
    -------
    pd.DataFrame
        Combined DataFrame of all matching CSV files, with a
        'source_file' column indicating which file each row came from.
    """
    file_paths = sorted(glob.glob(os.path.join(folder_path, pattern)))
 
    if not file_paths:
        raise FileNotFoundError(
            f"No files matching pattern '{pattern}' found in '{folder_path}'"
        )
 
    dataframes = []
    for path in file_paths:
        df = pd.read_csv(path)
        # Strip any stray whitespace/quotes from column names
        df.columns = [c.strip() for c in df.columns]
        df["source_file"] = os.path.basename(path)
        dataframes.append(df)
 
    combined_df = pd.concat(dataframes, ignore_index=True)
    return combined_df

In [4]:
def convert_date_column(
    df: pd.DataFrame,
    date_column: str = "Date",
    set_as_index: bool = True,
) -> pd.DataFrame:
    """
    Convert the date column to a proper datetime dtype (and optionally
    set it as the DataFrame index) to enable time-series analysis.
 
    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame containing a date column (e.g. '1/1/1979').
        (Weather data is given in the form of m/d/YYYY format)
    date_column : str, optional
        Name of the column holding date strings (default 'Date').
    set_as_index : bool, optional
        If True (default), sets the parsed date column as the
        DataFrame index and sorts by it, which is convenient for
        resampling, rolling windows, etc. If False, the column is
        converted in place but left as a regular column.
 
    Returns
    -------
    pd.DataFrame
        DataFrame with the date column converted to datetime64[ns].
    """
    df = df.copy()
    df[date_column] = pd.to_datetime(df[date_column], format="%m/%d/%Y", errors="coerce")
 
    if df[date_column].isna().any():
        n_bad = df[date_column].isna().sum()
        print(f"Warning: {n_bad} rows had unparseable dates and were set to NaT.")
 
    if set_as_index:
        df = df.set_index(date_column).sort_index()
    else:
        df = df.sort_values(date_column)
 
    return df

In [5]:
def filter_by_lat_lon(
    df: pd.DataFrame,
    lat_lon_pairs: Sequence[Tuple[float, float]],
    lat_column: str = "Latitude",
    lon_column: str = "Longitude",
    tolerance: float = 1e-3,
) -> pd.DataFrame:
    """
    Return only the records matching a given set of (Latitude, Longitude)
    pairs.
 
    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame containing latitude/longitude columns.
    lat_lon_pairs : Sequence[Tuple[float, float]]
        Iterable of (latitude, longitude) pairs to filter on, e.g.
        [(68.222, -147.812), (68.222, -148.125)].
    lat_column, lon_column : str, optional
        Column names for latitude and longitude (defaults 'Latitude',
        'Longitude').
    tolerance : float, optional
        Allowed absolute difference when matching coordinates, to
        account for floating point precision differences between
        files (default 1e-3 degrees).
 
    Returns
    -------
    pd.DataFrame
        Subset of df whose (lat, lon) is within `tolerance` of any of
        the requested pairs.
    """
    if not lat_lon_pairs:
        raise ValueError("lat_lon_pairs must contain at least one (lat, lon) tuple.")
 
    mask = pd.Series(False, index=df.index)
    for lat, lon in lat_lon_pairs:
        pair_mask = (
            (df[lat_column] - lat).abs().le(tolerance)
            & (df[lon_column] - lon).abs().le(tolerance)
        )
        mask |= pair_mask
 
    return df[mask].copy()
 

In [6]:
def filter_by_date_ranges(
    df: pd.DataFrame,
    date_ranges: Sequence[Tuple[str, str]],
    date_column: str = None,
) -> pd.DataFrame:
    """
    Return only the records that fall within one or more date ranges.
 
    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame. Either its index is a DatetimeIndex (as
        produced by `convert_date_column` with set_as_index=True), or
        `date_column` names a datetime column.
    date_ranges : Sequence[Tuple[str, str]]
        Iterable of (start_date, end_date) pairs, e.g.
        [("1979-01-01", "1979-01-31"), ("1980-06-01", "1980-06-30")].
        Dates are inclusive on both ends and can be any string
        pd.Timestamp can parse.
    date_column : str, optional
        Name of a datetime column to filter on. If None (default),
        the DataFrame's index is assumed to be the date (DatetimeIndex).
 
    Returns
    -------
    pd.DataFrame
        Subset of df whose date falls within any of the given ranges.
    """
    if not date_ranges:
        raise ValueError("date_ranges must contain at least one (start, end) tuple.")
 
    if date_column is None:
        dates = df.index
        if not isinstance(dates, pd.DatetimeIndex):
            raise TypeError(
                "DataFrame index is not a DatetimeIndex. Either pass "
                "date_column=<col name>, or run convert_date_column() "
                "with set_as_index=True first."
            )
    else:
        dates = pd.to_datetime(df[date_column])
 
    mask = pd.Series(False, index=df.index)
    for start, end in date_ranges:
        start_ts = pd.Timestamp(start)
        end_ts = pd.Timestamp(end)
        range_mask = (dates >= start_ts) & (dates <= end_ts)
        mask |= pd.Series(range_mask, index=df.index)
 
    return df[mask].copy()
 

In [8]:
# Load Data
folder = "../Dataset/WeatherDataFromAcrossAlaska/"
df = load_weather_data(folder)

In [41]:
# Convert to date-time
df = convert_date_column(df, date_column="Date", set_as_index=True)

In [42]:
df

,Longitude,Latitude,Elevation,Max Temperature,Min Temperature,Precipitation,Wind,Relative Humidity,Solar,source_file
Date,,,,,,,,,,
1979-01-01,-147.811996,68.222000,1263,-8.969,-17.569,0.424004,3.818202,0.921171,0.000000,weatherdata-682-1478.csv
1979-01-01,-148.438004,68.222000,1274,-9.358,-16.751,0.253201,4.570372,0.914457,0.000000,weatherdata-682-1484.csv
1979-01-01,-148.750000,70.095299,33,-6.665,-12.737,0.000000,3.489858,0.874794,0.000000,weatherdata-701-1488.csv
1979-01-01,-148.750000,68.222000,1524,-9.709,-16.423,0.172520,4.768244,0.911448,0.000000,weatherdata-682-1488.csv
1979-01-01,-148.438004,70.095299,25,-6.243,-14.479,0.000000,3.284222,0.885805,0.000000,weatherdata-701-1484.csv
...,...,...,...,...,...,...,...,...,...,...
2014-07-31,-148.750000,68.222000,1524,14.120,6.389,0.545884,1.347854,0.572862,11.065176,weatherdata-682-1488.csv
2014-07-31,-147.811996,70.095299,27,14.413,2.901,0.000000,4.595429,0.652516,11.518776,weatherdata-701-1478.csv
2014-07-31,-148.750000,69.470901,195,19.841,8.555,0.010300,1.713970,0.536870,12.044375,weatherdata-695-1488.csv


In [22]:
# Extract data based on coordinates (latitude-longitude pairs)
subset_coords = filter_by_lat_lon(
    df,
    lat_lon_pairs=[(68.222, -147.812), (68.222, -148.125)],
)

In [33]:
# Extract data based on date ranges (Use this format: yyyy-mm-dd)
# Use subset_coords instead of df for data from specific weatherstations (the ones extracted at subset_coords) for specific date ranges
subset_dates = filter_by_date_ranges(
    df,
    date_ranges=[("1979-12-20", "1979-12-21"), ("1979-06-01", "1979-06-02")],
)

In [44]:
training_data_weather = df
training_data_weather.to_csv('training_data_weather.csv')